# Resume the bird detector in plain Keras and export for ESP32 TFLite Micro

This notebook replaces the abandoned **TFLite Model Maker / EfficientDet-Lite1**
portion of `train_bird_detector_tflite.ipynb`. It starts from artifacts already
written by that notebook, so sections 1-17 do not need to be rerun.

It reuses:

- `joined_bird_detector_annotations.csv` and `data_audit_summary.json`;
- the staged Pascal VOC `train`, `validation`, `test`, and `calibration` splits;
- the ESP32-only calibration images;
- existing YOLO checkpoints/exports for optional comparison.

It creates a compact, single-class detector using only maintained TensorFlow/Keras
APIs: an ImageNet-pretrained MobileNetV2 backbone, a small anchor head, full-integer
TFLite conversion, Python validation, and a C++ byte array.

## Important deployment contract

The model returns a raw `[grid_y, grid_x, anchor, 5]` tensor containing box parameters
and bird-objectness logits. Decoding and non-max suppression deliberately remain
outside the graph. This avoids exporting a TensorFlow combined-NMS/custom detection
op that is a poor fit for TFLite Micro. The generated metadata JSON records the exact
anchors and decode equations required by ESP32 firmware.

No Model Maker package or API is used anywhere in this notebook.


## 1. Environment

Use the project's Python 3.12 WSL kernel (or another TensorFlow-supported Python
environment). Install only the direct dependencies if they are not already present:

```bash
python -m pip install --upgrade pip
python -m pip install tensorflow pandas pillow matplotlib pyyaml
```

Do **not** install `tflite-model-maker`.


In [1]:
import os
from pathlib import Path

# Optional: set this to a run name or absolute path. When blank, the newest COMPLETE
# run is selected; empty/newer run directories are ignored.
SOURCE_RUN_OVERRIDE = os.getenv("BIRD_DETECTOR_SOURCE_RUN", "")

RUN_TRAINING = os.getenv("BIRD_KERAS_RUN_TRAINING", "1") == "1"
RUN_EXPORT = os.getenv("BIRD_KERAS_RUN_EXPORT", "1") == "1"
RUN_EVALUATION = os.getenv("BIRD_KERAS_RUN_EVALUATION", "1") == "1"
WRITE_C_ARRAY = os.getenv("BIRD_KERAS_WRITE_C_ARRAY", "1") == "1"

SEED = 1234
INPUT_SIZE = 192              # Multiple of 32; smaller than EfficientDet-Lite1 for ESP32.
BACKBONE_ALPHA = 0.35          # Small ImageNet-pretrained MobileNetV2 width multiplier.
FEATURE_LAYER = "block_13_expand_relu"  # Output stride 16: 12x12 for 192px input.
NUM_ANCHORS = 3
BATCH_SIZE = 16
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 32
FINE_TUNE_BACKBONE_LAYERS = 30
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 8

REPRESENTATIVE_IMAGES = 500
PRE_NMS_SCORE = 0.01
NMS_IOU = 0.45
MAX_DETECTIONS = 25
EVALUATION_IOU = 0.50
SMALL_BIRD_AREA_PX2 = 32 * 32
CONFIDENCE_GRID = tuple(round(value / 100, 2) for value in range(1, 100, 2))
MAX_FALSE_POSITIVES_PER_NEGATIVE_IMAGE = 0.10

assert INPUT_SIZE % 32 == 0
assert NUM_ANCHORS >= 1


## 2. Imports, portable paths, and completed-run discovery


In [2]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import random
import re
import sys
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath, PureWindowsPath
from typing import Any, Iterable, Iterator, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw

try:
    import tensorflow as tf
except ImportError as error:
    raise RuntimeError(
        "TensorFlow is required. Select the project's Python 3.12 WSL kernel and "
        "install the packages listed in section 1."
    ) from error

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    candidates += [candidate / "esp32_Classifier" for candidate in candidates]
    for candidate in candidates:
        marker = candidate / "Models" / "Multi_Object_Detector" / "dataset_dirs.txt"
        if marker.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find Models/Multi_Object_Detector/dataset_dirs.txt from {start}"
    )


def portable_path(raw_path: str | os.PathLike[str]) -> Path:
    raw = str(raw_path).strip().strip('"').strip("'")
    windows_path = PureWindowsPath(raw)
    if windows_path.drive:
        if os.name == "nt":
            return Path(raw)
        drive = windows_path.drive.rstrip(":").lower()
        return Path("/mnt") / drive / Path(*windows_path.parts[1:])
    posix_path = PurePosixPath(raw)
    if os.name == "nt" and len(posix_path.parts) >= 4 and posix_path.parts[:2] == ("/", "mnt"):
        drive = posix_path.parts[2]
        if len(drive) == 1 and drive.isalpha():
            return Path(f"{drive.upper()}:\\", *posix_path.parts[3:])
    return Path(raw)


def json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Mapping):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


PROJECT_ROOT = find_project_root(Path.cwd())
DETECTOR_DIR = PROJECT_ROOT / "Models" / "Multi_Object_Detector"
RUNS_DIR = DETECTOR_DIR / "runs"


def is_complete_source_run(run_dir: Path) -> bool:
    voc = run_dir / "datasets" / "pascal_voc"
    required = [
        run_dir / "joined_bird_detector_annotations.csv",
        run_dir / "data_audit_summary.json",
        voc / "JPEGImages" / "train",
        voc / "Annotations" / "train",
        voc / "JPEGImages" / "validation",
        voc / "Annotations" / "validation",
        voc / "JPEGImages" / "test",
        voc / "Annotations" / "test",
        voc / "JPEGImages" / "calibration",
        voc / "Annotations" / "calibration",
    ]
    return all(path.exists() for path in required)


def resolve_source_run() -> Path:
    if SOURCE_RUN_OVERRIDE:
        candidate = portable_path(SOURCE_RUN_OVERRIDE)
        if not candidate.is_absolute():
            candidate = RUNS_DIR / candidate
        candidate = candidate.resolve()
        if not is_complete_source_run(candidate):
            raise FileNotFoundError(f"Incomplete source run: {candidate}")
        return candidate
    candidates = [path for path in RUNS_DIR.iterdir() if path.is_dir() and is_complete_source_run(path)]
    if not candidates:
        raise FileNotFoundError(f"No completed detector runs found below {RUNS_DIR}")
    return max(
        candidates,
        key=lambda path: (path.joinpath("joined_bird_detector_annotations.csv").stat().st_mtime_ns, path.name),
    ).resolve()


SOURCE_RUN_DIR = resolve_source_run()
VOC_ROOT = SOURCE_RUN_DIR / "datasets" / "pascal_voc"
KERAS_RUN_DIR = SOURCE_RUN_DIR / "keras_micro_detector"
CHECKPOINT_DIR = KERAS_RUN_DIR / "checkpoints"
EXPORT_DIR = SOURCE_RUN_DIR / "exports" / "keras_micro_detector"
FIRMWARE_DIR = EXPORT_DIR / "firmware"
for directory in (KERAS_RUN_DIR, CHECKPOINT_DIR, EXPORT_DIR, FIRMWARE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

HEAD_WEIGHTS = CHECKPOINT_DIR / "head_complete.weights.h5"
BEST_WEIGHTS = CHECKPOINT_DIR / "fine_tune_complete.weights.h5"
HEAD_BACKUP_DIR = CHECKPOINT_DIR / "head_backup"
FINE_TUNE_BACKUP_DIR = CHECKPOINT_DIR / "fine_tune_backup"
ANCHORS_PATH = KERAS_RUN_DIR / "anchors.json"
FLOAT_TFLITE_PATH = EXPORT_DIR / "bird_detector_mobilenetv2_float32.tflite"
INT8_TFLITE_PATH = EXPORT_DIR / "bird_detector_mobilenetv2_uint8.tflite"

print("Project root:", PROJECT_ROOT)
print("Reusing completed run:", SOURCE_RUN_DIR)
print("Keras artifacts:", KERAS_RUN_DIR)
print("TensorFlow:", tf.__version__)
print("Python:", sys.version.split()[0], platform.platform())

CONFIG_FINGERPRINT_PAYLOAD = {
    "source_manifest_sha256": sha256_file(SOURCE_RUN_DIR / "joined_bird_detector_annotations.csv"),
    "input_size": INPUT_SIZE,
    "backbone_alpha": BACKBONE_ALPHA,
    "feature_layer": FEATURE_LAYER,
    "num_anchors": NUM_ANCHORS,
}
CONFIG_FINGERPRINT = hashlib.sha256(
    json.dumps(CONFIG_FINGERPRINT_PAYLOAD, sort_keys=True).encode("utf-8")
).hexdigest()
print("Configuration fingerprint:", CONFIG_FINGERPRINT)


I0000 00:00:1793026179.474592   16676 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793026179.926163   16676 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793026181.824641   16676 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Project root: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier
Reusing completed run: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z
Keras artifacts: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z/keras_micro_detector
TensorFlow: 2.21.0
Python: 3.12.15 Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.43
Configuration fingerprint: 422d7a7d8c0b253cbf5aca7c72916285a2cdea3029a04b524b56d13c0938fa87


## 3. Import and audit the staged Pascal VOC artifacts


In [3]:
def load_voc_split(voc_root: Path, split: str) -> list[dict[str, Any]]:
    image_dir = voc_root / "JPEGImages" / split
    annotation_dir = voc_root / "Annotations" / split
    records: list[dict[str, Any]] = []
    for xml_path in sorted(annotation_dir.glob("*.xml")):
        root = ET.parse(xml_path).getroot()
        filename = root.findtext("filename")
        size = root.find("size")
        if not filename or size is None:
            raise ValueError(f"Malformed VOC annotation: {xml_path}")
        width = int(size.findtext("width", "0"))
        height = int(size.findtext("height", "0"))
        boxes = []
        for obj in root.findall("object"):
            if obj.findtext("name", "").casefold() != "bird":
                continue
            bounds = obj.find("bndbox")
            if bounds is None:
                continue
            box = [
                float(bounds.findtext("xmin", "0")),
                float(bounds.findtext("ymin", "0")),
                float(bounds.findtext("xmax", "0")),
                float(bounds.findtext("ymax", "0")),
            ]
            if box[2] > box[0] and box[3] > box[1]:
                boxes.append(box)
        image_path = image_dir / filename
        records.append({
            "sample_id": xml_path.stem,
            "image_path": image_path,
            "width": width,
            "height": height,
            "boxes": np.asarray(boxes, dtype=np.float32).reshape((-1, 4)),
            "split": split,
        })
    return records


VOC_SPLITS = {
    split: load_voc_split(VOC_ROOT, split)
    for split in ("train", "validation", "test", "calibration")
}

audit_rows = []
for split, records in VOC_SPLITS.items():
    missing = [str(record["image_path"]) for record in records if not record["image_path"].is_file()]
    audit_rows.append({
        "split": split,
        "images": len(records),
        "positive_images": sum(bool(len(record["boxes"])) for record in records),
        "negative_images": sum(not len(record["boxes"]) for record in records),
        "boxes": sum(len(record["boxes"]) for record in records),
        "missing_images": len(missing),
    })
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} staged {split} images are missing; first missing file: {missing[0]}"
        )

artifact_audit = pd.DataFrame(audit_rows)
display(artifact_audit)
assert len(VOC_SPLITS["train"]) > 0
assert len(VOC_SPLITS["validation"]) > 0
assert len(VOC_SPLITS["test"]) > 0
assert 0 < len(VOC_SPLITS["calibration"]) <= REPRESENTATIVE_IMAGES


,split,images,positive_images,negative_images,boxes,missing_images
0,train,6759,5331,1428,6840,0
1,validation,621,451,170,578,0
2,test,312,208,104,240,0
3,calibration,500,253,247,325,0


## 4. Letterboxing, anchor fitting, and target encoding

Images remain full-scene. They are letterboxed to a square without cropping. Anchors
are learned from the imported training boxes and persisted, so a resumed run uses the
identical decode contract.


In [4]:
def letterbox_geometry(width: float, height: float, size: int = INPUT_SIZE) -> tuple[float, float, float]:
    scale = min(size / width, size / height)
    resized_width = round(width * scale)
    resized_height = round(height * scale)
    pad_x = (size - resized_width) / 2.0
    pad_y = (size - resized_height) / 2.0
    return scale, pad_x, pad_y


def normalized_letterbox_wh(records: Sequence[Mapping[str, Any]]) -> np.ndarray:
    values = []
    for record in records:
        scale, _, _ = letterbox_geometry(record["width"], record["height"])
        for xmin, ymin, xmax, ymax in record["boxes"]:
            values.append([(xmax - xmin) * scale / INPUT_SIZE, (ymax - ymin) * scale / INPUT_SIZE])
    result = np.asarray(values, dtype=np.float32)
    return result[(result > 0).all(axis=1)]


def wh_iou(box_wh: np.ndarray, anchors: np.ndarray) -> np.ndarray:
    intersection = np.minimum(box_wh[:, None, :], anchors[None, :, :]).prod(axis=-1)
    union = box_wh.prod(axis=-1)[:, None] + anchors.prod(axis=-1)[None, :] - intersection
    return intersection / np.maximum(union, 1e-9)


def fit_anchors(box_wh: np.ndarray, count: int, seed: int = SEED, iterations: int = 100) -> np.ndarray:
    if len(box_wh) < count:
        raise ValueError(f"Need at least {count} boxes to fit anchors")
    rng = np.random.default_rng(seed)
    centers = box_wh[rng.choice(len(box_wh), count, replace=False)].copy()
    for _ in range(iterations):
        assignments = wh_iou(box_wh, centers).argmax(axis=1)
        updated = centers.copy()
        for index in range(count):
            members = box_wh[assignments == index]
            if len(members):
                updated[index] = np.median(members, axis=0)
        if np.allclose(updated, centers, rtol=0, atol=1e-6):
            break
        centers = updated
    order = np.argsort(centers.prod(axis=1))
    return centers[order].astype(np.float32)


if ANCHORS_PATH.is_file():
    anchor_contract = json.loads(ANCHORS_PATH.read_text(encoding="utf-8"))
    if anchor_contract.get("config_fingerprint") != CONFIG_FINGERPRINT:
        raise RuntimeError(
            f"Anchor contract {ANCHORS_PATH} belongs to a different data/model configuration. "
            "Archive the old keras_micro_detector directory before changing architecture settings."
        )
    ANCHORS = np.asarray(anchor_contract["anchors"], dtype=np.float32)
else:
    training_wh = normalized_letterbox_wh(VOC_SPLITS["train"])
    ANCHORS = fit_anchors(training_wh, NUM_ANCHORS)
    ANCHORS_PATH.write_text(
        json.dumps({
            "config_fingerprint": CONFIG_FINGERPRINT,
            "input_size": INPUT_SIZE,
            "anchors": ANCHORS.tolist(),
        }, indent=2),
        encoding="utf-8",
    )

if ANCHORS.shape != (NUM_ANCHORS, 2):
    raise ValueError(f"Unexpected anchor shape: {ANCHORS.shape}")
print("Normalized anchors [width, height]:\n", ANCHORS)


Normalized anchors [width, height]:
 [[0.09843751 0.0859375 ]
 [0.16718751 0.1609375 ]
 [0.568      0.5375    ]]


In [5]:
AUTOTUNE = tf.data.AUTOTUNE


def record_generator(records: Sequence[Mapping[str, Any]]) -> Iterator[tuple[str, np.ndarray]]:
    for record in records:
        yield str(record["image_path"]), record["boxes"].astype(np.float32)


def load_and_letterbox(path: tf.Tensor, boxes: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.cast(image, tf.float32)
    height = tf.cast(tf.shape(image)[0], tf.float32)
    width = tf.cast(tf.shape(image)[1], tf.float32)
    scale = tf.minimum(INPUT_SIZE / width, INPUT_SIZE / height)
    new_width = tf.cast(tf.round(width * scale), tf.int32)
    new_height = tf.cast(tf.round(height * scale), tf.int32)
    resized = tf.image.resize(image, [new_height, new_width], antialias=True)
    pad_left = (INPUT_SIZE - new_width) // 2
    pad_top = (INPUT_SIZE - new_height) // 2
    pad_right = INPUT_SIZE - new_width - pad_left
    pad_bottom = INPUT_SIZE - new_height - pad_top
    image = tf.pad(resized, [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]], constant_values=114.0)
    image.set_shape([INPUT_SIZE, INPUT_SIZE, 3])

    scale_vector = tf.stack([scale, scale, scale, scale])
    pad_vector = tf.cast(tf.stack([pad_left, pad_top, pad_left, pad_top]), tf.float32)
    boxes = (boxes * scale_vector + pad_vector) / float(INPUT_SIZE)
    boxes = tf.clip_by_value(boxes, 0.0, 1.0)
    return image, boxes


def augment(image: tf.Tensor, boxes: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    flip = tf.random.uniform(()) < 0.5
    image = tf.cond(flip, lambda: tf.image.flip_left_right(image), lambda: image)

    def flip_boxes() -> tf.Tensor:
        xmin, ymin, xmax, ymax = tf.unstack(boxes, axis=-1)
        return tf.stack([1.0 - xmax, ymin, 1.0 - xmin, ymax], axis=-1)

    boxes = tf.cond(flip, flip_boxes, lambda: boxes)
    image = tf.image.random_brightness(image, max_delta=12.0)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    return tf.clip_by_value(image, 0.0, 255.0), boxes


def encode_targets(boxes: tf.Tensor) -> tf.Tensor:
    target = tf.zeros([GRID_SIZE, GRID_SIZE, NUM_ANCHORS, 5], tf.float32)
    count = tf.shape(boxes)[0]

    def with_boxes() -> tf.Tensor:
        xmin, ymin, xmax, ymax = tf.unstack(boxes, axis=-1)
        width = tf.maximum(xmax - xmin, 1e-6)
        height = tf.maximum(ymax - ymin, 1e-6)
        center_x = tf.clip_by_value((xmin + xmax) / 2.0, 0.0, 1.0 - 1e-6)
        center_y = tf.clip_by_value((ymin + ymax) / 2.0, 0.0, 1.0 - 1e-6)
        wh = tf.stack([width, height], axis=-1)
        anchor_tensor = tf.constant(ANCHORS, tf.float32)
        intersection = tf.reduce_prod(tf.minimum(wh[:, None, :], anchor_tensor[None, :, :]), axis=-1)
        union = tf.reduce_prod(wh, axis=-1)[:, None] + tf.reduce_prod(anchor_tensor, axis=-1)[None, :] - intersection
        anchor_iou = intersection / tf.maximum(union, 1e-9)
        cell_x = tf.cast(tf.floor(center_x * GRID_SIZE), tf.int32)
        cell_y = tf.cast(tf.floor(center_y * GRID_SIZE), tf.int32)
        updates = tf.stack([center_x, center_y, width, height, tf.ones_like(center_x)], axis=-1)
        # Process the best anchor matches first. If two birds occupy the same grid
        # cell, the later bird receives its next-best free anchor instead of silently
        # overwriting an existing target.
        order = tf.argsort(tf.reduce_max(anchor_iou, axis=-1), direction="DESCENDING")
        ordered_iou = tf.gather(anchor_iou, order)
        ordered_x = tf.gather(cell_x, order)
        ordered_y = tf.gather(cell_y, order)
        ordered_updates = tf.gather(updates, order)

        def condition(index: tf.Tensor, current: tf.Tensor) -> tf.Tensor:
            return index < count

        def body(index: tf.Tensor, current: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
            ranked_anchors = tf.argsort(ordered_iou[index], direction="DESCENDING")
            cell_indices = tf.stack([
                tf.fill([NUM_ANCHORS], ordered_y[index]),
                tf.fill([NUM_ANCHORS], ordered_x[index]),
                ranked_anchors,
            ], axis=-1)
            occupied = tf.gather_nd(current[..., 4], cell_indices) > 0.5
            free_positions = tf.where(~occupied)

            def assign() -> tf.Tensor:
                chosen_position = tf.cast(free_positions[0, 0], tf.int32)
                chosen_anchor = ranked_anchors[chosen_position]
                chosen_index = tf.stack([
                    ordered_y[index], ordered_x[index], chosen_anchor
                ])[None, :]
                return tf.tensor_scatter_nd_update(
                    current, chosen_index, ordered_updates[index][None, :]
                )

            current = tf.cond(tf.shape(free_positions)[0] > 0, assign, lambda: current)
            return index + 1, current

        _, encoded = tf.while_loop(
            condition,
            body,
            (tf.constant(0), target),
            parallel_iterations=1,
        )
        return encoded

    return tf.cond(count > 0, with_boxes, lambda: target)


def make_dataset(
    records: Sequence[Mapping[str, Any]], batch_size: int, training: bool
) -> tf.data.Dataset:
    dataset = tf.data.Dataset.from_generator(
        lambda: record_generator(records),
        output_signature=(
            tf.TensorSpec([], tf.string),
            tf.TensorSpec([None, 4], tf.float32),
        ),
    )
    if training:
        dataset = dataset.shuffle(min(len(records), 4096), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_and_letterbox, num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.map(augment, num_parallel_calls=AUTOTUNE)
    dataset = dataset.map(
        lambda image, boxes: (image, encode_targets(boxes)),
        num_parallel_calls=AUTOTUNE,
    )
    return dataset.batch(batch_size).prefetch(AUTOTUNE)


## 5. Plain-Keras MobileNetV2 detector and loss


In [6]:
def build_detector() -> tuple[tf.keras.Model, tf.keras.Model]:
    base = tf.keras.applications.MobileNetV2(
        input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
        alpha=BACKBONE_ALPHA,
        include_top=False,
        weights="imagenet",
    )
    feature_extractor = tf.keras.Model(
        base.input, base.get_layer(FEATURE_LAYER).output, name="mobilenetv2_features"
    )
    inputs = tf.keras.Input((INPUT_SIZE, INPUT_SIZE, 3), name="image", dtype=tf.float32)
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="normalize_uint8_rgb")(inputs)
    x = feature_extractor(x)
    x = tf.keras.layers.SeparableConv2D(96, 3, padding="same", use_bias=False, name="detector_sepconv")(x)
    x = tf.keras.layers.BatchNormalization(name="detector_bn")(x)
    x = tf.keras.layers.ReLU(max_value=6.0, name="detector_relu6")(x)
    x = tf.keras.layers.Conv2D(NUM_ANCHORS * 5, 1, padding="same", name="detector_logits")(x)
    grid_height = int(x.shape[1])
    grid_width = int(x.shape[2])
    if grid_height != grid_width:
        raise ValueError(f"Expected square feature grid, got {x.shape}")
    outputs = tf.keras.layers.Reshape(
        (grid_height, grid_width, NUM_ANCHORS, 5), name="raw_detections"
    )(x)
    return tf.keras.Model(inputs, outputs, name="bird_detector_mobilenetv2"), feature_extractor


model, feature_extractor = build_detector()
GRID_SIZE = int(model.output_shape[1])
print("Grid size:", GRID_SIZE, "x", GRID_SIZE)
model.summary()


def decode_raw_boxes(raw: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    grid_y, grid_x = tf.meshgrid(
        tf.range(GRID_SIZE, dtype=tf.float32),
        tf.range(GRID_SIZE, dtype=tf.float32),
        indexing="ij",
    )
    grid = tf.stack([grid_x, grid_y], axis=-1)[None, :, :, None, :]
    center = (tf.sigmoid(raw[..., 0:2]) + grid) / float(GRID_SIZE)
    size = tf.constant(ANCHORS, tf.float32)[None, None, None, :, :] * tf.exp(
        tf.clip_by_value(raw[..., 2:4], -4.0, 4.0)
    )
    return tf.concat([center, size], axis=-1), raw[..., 4]


def detector_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    true_box = y_true[..., 0:4]
    object_mask = y_true[..., 4]
    predicted_box, object_logits = decode_raw_boxes(y_pred)

    binary_cross_entropy = tf.nn.sigmoid_cross_entropy_with_logits(
        labels=object_mask, logits=object_logits
    )
    probability = tf.sigmoid(object_logits)
    probability_true = object_mask * probability + (1.0 - object_mask) * (1.0 - probability)
    alpha = object_mask * 0.75 + (1.0 - object_mask) * 0.25
    object_loss = tf.reduce_mean(alpha * tf.pow(1.0 - probability_true, 2.0) * binary_cross_entropy)

    coordinate_error = tf.keras.losses.Huber(reduction="none")(true_box, predicted_box)
    positive_count = tf.maximum(1.0, tf.reduce_sum(object_mask))
    box_loss = tf.reduce_sum(coordinate_error * object_mask) / positive_count
    return object_loss + 8.0 * box_loss


def mean_positive_objectness(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    mask = y_true[..., 4]
    values = tf.sigmoid(y_pred[..., 4]) * mask
    return tf.reduce_sum(values) / tf.maximum(1.0, tf.reduce_sum(mask))


def compile_detector(learning_rate: float) -> None:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=detector_loss,
        metrics=[mean_positive_objectness],
    )


I0000 00:00:1793026424.766635   16676 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Grid size: 12 x 12


Model: "bird_detector_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalize_uint8_rgb (Rescaling) │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_features            │ (None, 12, 12, 192)    │        98,512 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ detector_sepconv                │ (None, 12, 12, 96)     │        20,160 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ detector_bn                     │ (None, 12, 12, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ detector_relu6 (ReLU)           │ (None, 12, 12, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ detector_logits (Conv2D)        │ (None, 12, 12, 15)     │         1,455 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ raw_detections (Reshape)        │ (None, 12, 12, 3, 5)   │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 120,511 (470.75 KB)

 Trainable params: 113,775 (444.43 KB)

 Non-trainable params: 6,736 (26.31 KB)

## 6. Resume-aware transfer learning

Completed stages are persisted separately. A rerun skips the frozen-head stage after
`head_complete.weights.h5` exists and skips all fitting after
`fine_tune_complete.weights.h5` exists. Keras `BackupAndRestore` also resumes an
interrupted `fit()` call with its optimizer state. Set
`BIRD_KERAS_RUN_TRAINING=0` to skip fitting and only export/evaluate an existing
checkpoint.


In [7]:
def callbacks(stage: str, append_log: bool) -> list[tf.keras.callbacks.Callback]:
    backup_dir = HEAD_BACKUP_DIR if stage == "head" else FINE_TUNE_BACKUP_DIR
    return [
        tf.keras.callbacks.BackupAndRestore(backup_dir=backup_dir),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(KERAS_RUN_DIR / "training_log.csv", append=append_log),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


def train_or_resume() -> int:
    try:
        train_dataset = make_dataset(VOC_SPLITS["train"], BATCH_SIZE, training=True)
        validation_dataset = make_dataset(VOC_SPLITS["validation"], BATCH_SIZE, training=False)

        if BEST_WEIGHTS.is_file():
            model.load_weights(BEST_WEIGHTS)
            print("Fine tuning already complete; restored", BEST_WEIGHTS)
            return BATCH_SIZE

        if HEAD_WEIGHTS.is_file():
            model.load_weights(HEAD_WEIGHTS)
            print("Frozen-head stage already complete; restored", HEAD_WEIGHTS)
        else:
            feature_extractor.trainable = False
            compile_detector(HEAD_LEARNING_RATE)
            model.fit(
                train_dataset,
                validation_data=validation_dataset,
                epochs=HEAD_EPOCHS,
                callbacks=callbacks(stage="head", append_log=False),
            )
            model.save_weights(HEAD_WEIGHTS)
            print("Saved completed head stage:", HEAD_WEIGHTS)

        feature_extractor.trainable = True
        for layer in feature_extractor.layers[:-FINE_TUNE_BACKBONE_LAYERS]:
            layer.trainable = False
        for layer in feature_extractor.layers[-FINE_TUNE_BACKBONE_LAYERS:]:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
        compile_detector(FINE_TUNE_LEARNING_RATE)
        model.fit(
            train_dataset,
            validation_data=validation_dataset,
            epochs=FINE_TUNE_EPOCHS,
            callbacks=callbacks(stage="fine_tune", append_log=True),
        )
        model.save_weights(BEST_WEIGHTS)
        print("Saved completed fine-tune stage:", BEST_WEIGHTS)
        return BATCH_SIZE
    except tf.errors.ResourceExhaustedError:
        raise RuntimeError(
            f"Out of memory at batch {BATCH_SIZE}. Reduce BATCH_SIZE in section 1 "
            "and rerun sections 5-6; BackupAndRestore will resume the active stage."
        ) from None


if RUN_TRAINING:
    effective_batch_size = train_or_resume()
    if BEST_WEIGHTS.is_file():
        model.load_weights(BEST_WEIGHTS)
    print("Deployment weights:", BEST_WEIGHTS)
else:
    effective_batch_size = None
    if not BEST_WEIGHTS.is_file():
        raise FileNotFoundError(
            f"Training is disabled and no checkpoint exists at {BEST_WEIGHTS}"
        )
    print("Training skipped; using", BEST_WEIGHTS)


Epoch 1/8


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793026450.634481   16880 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1793026451.480310   16785 service.cc:153] XLA service 0x76ed4c0327c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793026451.480364   16785 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793026451.604677   16785 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1

      5/Unknown 13s 30ms/step - loss: 0.1341 - mean_positive_objectness: 0.5615

I0000 00:00:1793026460.221389   16785 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion', 220 bytes spill stores, 216 bytes spill loads

I0000 00:00:1793026460.234021   16785 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    423/Unknown 43s 73ms/step - loss: 0.0383 - mean_positive_objectness: 0.2530

/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


... (13 outputs left out)

I0000 00:00:1793026621.534134   16875 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 10240006905470475188
I0000 00:00:1793026621.534195   16875 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 7547312831778504664


423/423 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss: 0.0075 - mean_positive_objectness: 0.3731 - val_loss: 0.0157 - val_mean_positive_objectness: 0.2061 - learning_rate: 0.0010
Saved completed head stage: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z/keras_micro_detector/checkpoints/head_complete.weights.h5
Epoch 1/32
      4/Unknown 10s 44ms/step - loss: 0.0045 - mean_positive_objectness: 0.3432

I0000 00:00:1793026634.282438   16784 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion', 220 bytes spill stores, 216 bytes spill loads



423/423 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - loss: 0.0078 - mean_positive_objectness: 0.3544 - val_loss: 0.0172 - val_mean_positive_objectness: 0.2148 - learning_rate: 1.0000e-04
Epoch 2/32
423/423 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - loss: 0.0067 - mean_positive_objectness: 0.3618 - val_loss: 0.0165 - val_mean_positive_objectness: 0.2071 - learning_rate: 1.0000e-04
Epoch 3/32
423/423 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - loss: 0.0062 - mean_positive_objectness: 0.3685 - val_loss: 0.0164 - val_mean_positive_objectness: 0.2194 - learning_rate: 1.0000e-04
Epoch 4/32
423/423 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - loss: 0.0056 - mean_positive_objectness: 0.3763 - val_loss: 0.0148 - val_mean_positive_objectness: 0.2232 - learning_rate: 1.0000e-04
Epoch 5/32
423/423 ━━━━━━━━━━━━━━━━━━━━ 16s 36ms/step - loss: 0.0055 - mean_positive_objectness: 0.3824 - val_loss: 0.0150 - val_mean_positive_objectness: 0.2240 - learning_rate: 1.0000e-04
Epoch 6/32
423/423 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - loss:

## 7. Float and full-integer TFLite export


In [8]:
def representative_dataset() -> Iterator[list[np.ndarray]]:
    records = VOC_SPLITS["calibration"][:REPRESENTATIVE_IMAGES]
    dataset = make_dataset(records, batch_size=1, training=False)
    for image, _ in dataset.unbatch().batch(1):
        yield [image.numpy().astype(np.float32)]


def convert_float_tflite(model: tf.keras.Model) -> bytes:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    return converter.convert()


def convert_full_integer_tflite(model: tf.keras.Model) -> bytes:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    return converter.convert()


def interpreter_details(path: Path) -> dict[str, Any]:
    interpreter = tf.lite.Interpreter(model_path=str(path))
    interpreter.allocate_tensors()
    inputs = interpreter.get_input_details()
    outputs = interpreter.get_output_details()
    operations = []
    if hasattr(interpreter, "_get_ops_details"):
        operations = sorted({
            item["op_name"]
            for item in interpreter._get_ops_details()
            if item["op_name"] != "DELEGATE"
        })
    return {
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "inputs": [
            {
                "name": item["name"],
                "shape": item["shape"].tolist(),
                "dtype": np.dtype(item["dtype"]).name,
                "quantization": list(item["quantization"]),
            }
            for item in inputs
        ],
        "outputs": [
            {
                "name": item["name"],
                "shape": item["shape"].tolist(),
                "dtype": np.dtype(item["dtype"]).name,
                "quantization": list(item["quantization"]),
            }
            for item in outputs
        ],
        "operations": operations,
    }


if RUN_EXPORT:
    if BEST_WEIGHTS.is_file():
        model.load_weights(BEST_WEIGHTS)
    FLOAT_TFLITE_PATH.write_bytes(convert_float_tflite(model))
    INT8_TFLITE_PATH.write_bytes(convert_full_integer_tflite(model))
    export_details = {
        "float": interpreter_details(FLOAT_TFLITE_PATH),
        "uint8": interpreter_details(INT8_TFLITE_PATH),
    }
    (EXPORT_DIR / "tflite_tensor_details.json").write_text(
        json.dumps(json_safe(export_details), indent=2), encoding="utf-8"
    )
    (EXPORT_DIR / "export_contract.json").write_text(
        json.dumps({
            "config_fingerprint": CONFIG_FINGERPRINT,
            "anchors_sha256": sha256_file(ANCHORS_PATH),
            "weights_sha256": sha256_file(BEST_WEIGHTS),
            "float_tflite_sha256": sha256_file(FLOAT_TFLITE_PATH),
            "uint8_tflite_sha256": sha256_file(INT8_TFLITE_PATH),
        }, indent=2),
        encoding="utf-8",
    )
    display(pd.DataFrame([
        {
            "precision": name,
            "size_mb": details["bytes"] / 1_000_000,
            "input_dtype": details["inputs"][0]["dtype"],
            "output_dtype": details["outputs"][0]["dtype"],
            "operations": ", ".join(details["operations"]),
        }
        for name, details in export_details.items()
    ]))
    assert len(export_details["uint8"]["inputs"]) == 1
    assert len(export_details["uint8"]["outputs"]) == 1
    assert export_details["uint8"]["inputs"][0]["dtype"] == "uint8"
    assert export_details["uint8"]["outputs"][0]["dtype"] == "uint8"
    assert export_details["uint8"]["inputs"][0]["shape"] == [1, INPUT_SIZE, INPUT_SIZE, 3]
    assert export_details["uint8"]["outputs"][0]["shape"] == [1, GRID_SIZE, GRID_SIZE, NUM_ANCHORS, 5]
    assert export_details["uint8"]["inputs"][0]["quantization"][0] > 0
    assert export_details["uint8"]["outputs"][0]["quantization"][0] > 0
else:
    for path in (FLOAT_TFLITE_PATH, INT8_TFLITE_PATH):
        if not path.is_file():
            raise FileNotFoundError(f"Export disabled and model is missing: {path}")
    export_contract = json.loads((EXPORT_DIR / "export_contract.json").read_text(encoding="utf-8"))
    if export_contract.get("config_fingerprint") != CONFIG_FINGERPRINT:
        raise RuntimeError("Existing TFLite exports belong to a different model/data configuration")
    export_details = json.loads((EXPORT_DIR / "tflite_tensor_details.json").read_text(encoding="utf-8"))
    print("Export skipped; existing TFLite models found.")


INFO:tensorflow:Assets written to: /tmp/tmp8wi19wpi/assets


INFO:tensorflow:Assets written to: /tmp/tmp8wi19wpi/assets


Saved artifact at '/tmp/tmp8wi19wpi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 192, 192, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 12, 12, 3, 5), dtype=tf.float32, name=None)
Captures:
  130768146807312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910765200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910766160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130768146807696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130768146807504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130768146807120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910766352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910767120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910766736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910765968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1307679107

W0000 00:00:1793027170.018051   16676 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1793027170.018106   16676 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1793027170.018492   16676 reader.cc:83] Reading SavedModel from: /tmp/tmp8wi19wpi
I0000 00:00:1793027170.025535   16676 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1793027170.025564   16676 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp8wi19wpi
I0000 00:00:1793027170.088752   16676 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1793027170.098095   16676 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1793027170.489191   16676 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp8wi19wpi
I0000 00:00:1793027170.590969   16676 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 572490 microseconds.


INFO:tensorflow:Assets written to: /tmp/tmpasyu5bh7/assets


INFO:tensorflow:Assets written to: /tmp/tmpasyu5bh7/assets


Saved artifact at '/tmp/tmpasyu5bh7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 192, 192, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 12, 12, 3, 5), dtype=tf.float32, name=None)
Captures:
  130768146807312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910765200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910766160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130768146807696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130768146807504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130768146807120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910766352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910767120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910766736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130767910765968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1307679107

/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1793027176.047981   16676 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1793027176.048022   16676 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1793027176.048175   16676 reader.cc:83] Reading SavedModel from: /tmp/tmpasyu5bh7
I0000 00:00:1793027176.054973   16676 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1793027176.055037   16676 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpasyu5bh7
I0000 00:00:1793027176.118327   16676 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1793027176.473175   16676 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpasyu5bh7
I0000 00:00:1793027176.555067   16676 loader.cc:471] SavedModel load for ta

,precision,size_mb,input_dtype,output_dtype,operations
0,float,0.496120,float32,float32,"ADD, CONV_2D, DEPTHWISE_CONV_2D, MUL, PACK, RE..."
1,uint8,0.265296,uint8,uint8,"ADD, CONV_2D, DEPTHWISE_CONV_2D, MUL, PACK, QU..."


## 8. Generate the ESP32 C++ array and decoder metadata


In [9]:
def write_cpp_array(
    binary_path: Path,
    header_path: Path,
    source_path: Path,
    symbol: str,
) -> None:
    data = binary_path.read_bytes()
    header_guard = re.sub(r"[^A-Z0-9]+", "_", header_path.name.upper())
    header_path.write_text(
        "\n".join([
            f"#ifndef {header_guard}",
            f"#define {header_guard}",
            "",
            "#include <cstddef>",
            "#include <cstdint>",
            "",
            f"extern const std::uint8_t {symbol}[];",
            f"extern const std::size_t {symbol}_len;",
            "",
            f"#endif  // {header_guard}",
            "",
        ]),
        encoding="utf-8",
    )
    lines = []
    for offset in range(0, len(data), 12):
        chunk = data[offset:offset + 12]
        lines.append("    " + ", ".join(f"0x{value:02x}" for value in chunk) + ",")
    source_path.write_text(
        "\n".join([
            f'#include "{header_path.name}"',
            "",
            f"alignas(16) const std::uint8_t {symbol}[] = {{",
            *lines,
            "};",
            f"const std::size_t {symbol}_len = sizeof({symbol});",
            "",
        ]),
        encoding="utf-8",
    )


DECODER_METADATA_PATH = FIRMWARE_DIR / "bird_detector_metadata.json"
uint8_input_contract = export_details["uint8"]["inputs"][0]
uint8_output_contract = export_details["uint8"]["outputs"][0]


def write_decoder_metadata(deployment_score_threshold: float | None) -> None:
    decoder_metadata = {
        "schema_version": 1,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "model": INT8_TFLITE_PATH.name,
        "model_sha256": sha256_file(INT8_TFLITE_PATH),
        "class_map": {"0": "bird"},
        "input": {
            **uint8_input_contract,
            "color_order": "RGB",
            "geometry": "letterbox",
            "pad_value": 114,
        },
        "output": {
            **uint8_output_contract,
            "order": ["tx", "ty", "tw", "th", "objectness_logit"],
            "anchors_normalized_wh": ANCHORS.tolist(),
            "decode": {
                "dequantize": "real_value = (quantized_value - zero_point) * scale",
                "center_x": "(sigmoid(tx) + grid_x) / grid_width",
                "center_y": "(sigmoid(ty) + grid_y) / grid_height",
                "width": "anchor_width * exp(clamp(tw, -4, 4))",
                "height": "anchor_height * exp(clamp(th, -4, 4))",
                "score": "sigmoid(objectness_logit)",
            },
        },
        "postprocess": {
            "candidate_score_threshold": PRE_NMS_SCORE,
            "deployment_score_threshold": deployment_score_threshold,
            "deployment_threshold_source": (
                "ESP32 validation recall-first sweep"
                if deployment_score_threshold is not None
                else "not evaluated yet"
            ),
            "nms_iou": NMS_IOU,
            "max_detections": MAX_DETECTIONS,
        },
        "operations": export_details["uint8"]["operations"],
    }
    DECODER_METADATA_PATH.write_text(
        json.dumps(decoder_metadata, indent=2), encoding="utf-8"
    )


if WRITE_C_ARRAY:
    write_cpp_array(
        INT8_TFLITE_PATH,
        FIRMWARE_DIR / "bird_detector_model.h",
        FIRMWARE_DIR / "bird_detector_model.cc",
        "g_bird_detector_model",
    )
    write_decoder_metadata(deployment_score_threshold=None)
    print("Firmware model array:", FIRMWARE_DIR / "bird_detector_model.cc")
    print("Decoder metadata:", DECODER_METADATA_PATH)


Firmware model array: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z/exports/keras_micro_detector/firmware/bird_detector_model.cc
Decoder metadata: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z/exports/keras_micro_detector/firmware/bird_detector_metadata.json


## 9. TFLite inference adapter with host-side decode and NMS


In [10]:
def sigmoid_np(value: np.ndarray) -> np.ndarray:
    value = np.clip(value, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-value))


def quantize_array(array: np.ndarray, detail: Mapping[str, Any]) -> np.ndarray:
    dtype = detail["dtype"]
    if dtype in (np.uint8, np.int8):
        scale, zero_point = detail["quantization"]
        if scale <= 0:
            raise ValueError(f"Invalid quantization parameters: {detail['quantization']}")
        limits = np.iinfo(dtype)
        return np.clip(np.round(array / scale + zero_point), limits.min, limits.max).astype(dtype)
    return array.astype(dtype)


def dequantize_array(array: np.ndarray, detail: Mapping[str, Any]) -> np.ndarray:
    if detail["dtype"] in (np.uint8, np.int8):
        scale, zero_point = detail["quantization"]
        return (array.astype(np.float32) - zero_point) * scale
    return array.astype(np.float32)


def numpy_letterbox(image: Image.Image) -> tuple[np.ndarray, float, float, float]:
    original_width, original_height = image.size
    scale, pad_x_float, pad_y_float = letterbox_geometry(original_width, original_height)
    resized_width = round(original_width * scale)
    resized_height = round(original_height * scale)
    pad_left = (INPUT_SIZE - resized_width) // 2
    pad_top = (INPUT_SIZE - resized_height) // 2
    canvas = np.full((INPUT_SIZE, INPUT_SIZE, 3), 114, dtype=np.uint8)
    resized = np.asarray(
        image.resize((resized_width, resized_height), Image.Resampling.BILINEAR),
        dtype=np.uint8,
    )
    canvas[pad_top:pad_top + resized_height, pad_left:pad_left + resized_width] = resized
    return canvas, scale, float(pad_left), float(pad_top)


def box_iou_array(box: np.ndarray, boxes: np.ndarray) -> np.ndarray:
    left_top = np.maximum(box[:2], boxes[:, :2])
    right_bottom = np.minimum(box[2:], boxes[:, 2:])
    intersection = np.maximum(0.0, right_bottom - left_top).prod(axis=1)
    box_area = np.maximum(0.0, box[2:] - box[:2]).prod()
    boxes_area = np.maximum(0.0, boxes[:, 2:] - boxes[:, :2]).prod(axis=1)
    return intersection / np.maximum(box_area + boxes_area - intersection, 1e-9)


def numpy_nms(boxes: np.ndarray, scores: np.ndarray, threshold: float, limit: int) -> np.ndarray:
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) and len(keep) < limit:
        current = int(order[0])
        keep.append(current)
        if len(order) == 1:
            break
        remaining = order[1:]
        order = remaining[box_iou_array(boxes[current], boxes[remaining]) < threshold]
    return np.asarray(keep, dtype=np.int32)


def decode_output(
    raw: np.ndarray,
    original_width: int,
    original_height: int,
    scale: float,
    pad_x: float,
    pad_y: float,
    score_threshold: float = PRE_NMS_SCORE,
) -> tuple[np.ndarray, np.ndarray]:
    grid_height, grid_width, anchor_count, _ = raw.shape
    grid_y, grid_x = np.meshgrid(np.arange(grid_height), np.arange(grid_width), indexing="ij")
    center_x = (sigmoid_np(raw[..., 0]) + grid_x[..., None]) / grid_width
    center_y = (sigmoid_np(raw[..., 1]) + grid_y[..., None]) / grid_height
    width = ANCHORS[None, None, :, 0] * np.exp(np.clip(raw[..., 2], -4.0, 4.0))
    height = ANCHORS[None, None, :, 1] * np.exp(np.clip(raw[..., 3], -4.0, 4.0))
    scores = sigmoid_np(raw[..., 4])
    selected = scores >= score_threshold
    if not selected.any():
        return np.empty((0, 4), np.float32), np.empty((0,), np.float32)
    boxes = np.stack([
        center_x - width / 2.0,
        center_y - height / 2.0,
        center_x + width / 2.0,
        center_y + height / 2.0,
    ], axis=-1)[selected]
    scores = scores[selected]
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] * INPUT_SIZE - pad_x) / scale
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] * INPUT_SIZE - pad_y) / scale
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, original_width)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, original_height)
    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    boxes, scores = boxes[valid], scores[valid]
    if not len(boxes):
        return boxes, scores
    keep = numpy_nms(boxes, scores, NMS_IOU, MAX_DETECTIONS)
    return boxes[keep], scores[keep]


def predict_tflite(model_path: Path, records: Sequence[Mapping[str, Any]]) -> pd.DataFrame:
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]
    predictions = []
    for record in records:
        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")
            tensor, scale, pad_x, pad_y = numpy_letterbox(image)
        interpreter.set_tensor(
            input_detail["index"],
            quantize_array(tensor[None].astype(np.float32), input_detail),
        )
        interpreter.invoke()
        raw = dequantize_array(interpreter.get_tensor(output_detail["index"]), output_detail)[0]
        boxes, scores = decode_output(
            raw, record["width"], record["height"], scale, pad_x, pad_y
        )
        for box, score in zip(boxes, scores):
            predictions.append({
                "sample_id": record["sample_id"],
                "xmin": float(box[0]), "ymin": float(box[1]),
                "xmax": float(box[2]), "ymax": float(box[3]),
                "score": float(score),
            })
    return pd.DataFrame(
        predictions,
        columns=["sample_id", "xmin", "ymin", "xmax", "ymax", "score"],
    )


## 10. Recall-first ESP32 validation and untouched test evaluation


In [11]:
def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    left, top = max(a[0], b[0]), max(a[1], b[1])
    right, bottom = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0.0, right - left) * max(0.0, bottom - top)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def projected_area(box: Sequence[float], width: int, height: int) -> float:
    scale, _, _ = letterbox_geometry(width, height)
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1]) * scale * scale


def evaluate_predictions(
    records: Sequence[Mapping[str, Any]],
    predictions: pd.DataFrame,
    confidence_threshold: float,
) -> dict[str, Any]:
    true_positives = false_positives = false_negatives = 0
    small_true_positives = small_false_negatives = 0
    false_positives_on_negative_images = 0
    negative_images = sum(not len(record["boxes"]) for record in records)
    matched_ious = []
    for record in records:
        ground_truth = record["boxes"]
        selected = predictions.loc[
            predictions["sample_id"].eq(record["sample_id"])
            & predictions["score"].ge(confidence_threshold)
        ].sort_values("score", ascending=False)
        matched: set[int] = set()
        for prediction in selected.itertuples(index=False):
            best_index = None
            best_iou = 0.0
            for index, truth in enumerate(ground_truth):
                if index in matched:
                    continue
                iou = box_iou(
                    (prediction.xmin, prediction.ymin, prediction.xmax, prediction.ymax), truth
                )
                if iou > best_iou:
                    best_iou, best_index = iou, index
            if best_index is not None and best_iou >= EVALUATION_IOU:
                matched.add(best_index)
                true_positives += 1
                matched_ious.append(best_iou)
                if projected_area(ground_truth[best_index], record["width"], record["height"]) < SMALL_BIRD_AREA_PX2:
                    small_true_positives += 1
            else:
                false_positives += 1
                if not len(ground_truth):
                    false_positives_on_negative_images += 1
        for index, truth in enumerate(ground_truth):
            if index not in matched:
                false_negatives += 1
                if projected_area(truth, record["width"], record["height"]) < SMALL_BIRD_AREA_PX2:
                    small_false_negatives += 1

    precision = true_positives / max(1, true_positives + false_positives)
    recall = true_positives / max(1, true_positives + false_negatives)
    f2 = 5.0 * precision * recall / max(1e-12, 4.0 * precision + recall)
    return {
        "confidence_threshold": confidence_threshold,
        "iou_threshold": EVALUATION_IOU,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f2": f2,
        "small_bird_recall": small_true_positives / max(1, small_true_positives + small_false_negatives),
        "negative_images": negative_images,
        "false_positives_on_negative_images": false_positives_on_negative_images,
        "false_positives_per_negative_image": false_positives_on_negative_images / max(1, negative_images),
        "mean_matched_iou": float(np.mean(matched_ious)) if matched_ious else 0.0,
    }


def select_threshold(
    records: Sequence[Mapping[str, Any]], predictions: pd.DataFrame
) -> tuple[float, pd.DataFrame]:
    sweep = pd.DataFrame([
        evaluate_predictions(records, predictions, threshold)
        for threshold in CONFIDENCE_GRID
    ])
    feasible = sweep.loc[
        sweep["false_positives_per_negative_image"].le(MAX_FALSE_POSITIVES_PER_NEGATIVE_IMAGE)
    ]
    if feasible.empty:
        raise RuntimeError(
            "No confidence threshold in 0.01-0.99 satisfies the configured "
            "false-positive-per-negative-image constraint."
        )
    best = feasible.sort_values(
        ["small_bird_recall", "recall", "f2", "precision", "confidence_threshold"],
        ascending=[False, False, False, False, True],
    ).iloc[0]
    return float(best["confidence_threshold"]), sweep


def compare_prediction_drift(float_predictions: pd.DataFrame, int8_predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    sample_ids = sorted(set(float_predictions["sample_id"]) | set(int8_predictions["sample_id"]))
    for sample_id in sample_ids:
        float_rows = float_predictions.loc[float_predictions["sample_id"].eq(sample_id)].sort_values("score", ascending=False)
        int8_rows = int8_predictions.loc[int8_predictions["sample_id"].eq(sample_id)].sort_values("score", ascending=False)
        used: set[int] = set()
        for float_row in float_rows.itertuples(index=False):
            best_index = None
            best_iou = 0.0
            for index, int8_row in int8_rows.iterrows():
                if index in used:
                    continue
                iou = box_iou(
                    (float_row.xmin, float_row.ymin, float_row.xmax, float_row.ymax),
                    (int8_row.xmin, int8_row.ymin, int8_row.xmax, int8_row.ymax),
                )
                if iou > best_iou:
                    best_iou, best_index = iou, index
            if best_index is None or best_iou < EVALUATION_IOU:
                rows.append({"sample_id": sample_id, "status": "lost_after_quantization", "iou": 0.0})
            else:
                used.add(best_index)
                int8_row = int8_rows.loc[best_index]
                rows.append({
                    "sample_id": sample_id,
                    "status": "matched",
                    "iou": best_iou,
                    "float_score": float_row.score,
                    "int8_score": int8_row.score,
                    "score_delta": int8_row.score - float_row.score,
                })
        for index in int8_rows.index:
            if index not in used:
                rows.append({"sample_id": sample_id, "status": "new_after_quantization", "iou": 0.0})
    return pd.DataFrame(rows)


In [12]:
FINAL_RESULTS: list[dict[str, Any]] = []
SELECTED_THRESHOLDS: dict[str, float] = {}
if RUN_EVALUATION:
    model_paths = {
        "keras_mobilenetv2_float": FLOAT_TFLITE_PATH,
        "keras_mobilenetv2_uint8": INT8_TFLITE_PATH,
    }
    all_predictions: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}
    for model_name, model_path in model_paths.items():
        validation_predictions = predict_tflite(model_path, VOC_SPLITS["validation"])
        test_predictions = predict_tflite(model_path, VOC_SPLITS["test"])
        all_predictions[model_name] = (validation_predictions, test_predictions)
        threshold, sweep = select_threshold(VOC_SPLITS["validation"], validation_predictions)
        SELECTED_THRESHOLDS[model_name] = threshold
        sweep.assign(model=model_name).to_csv(
            KERAS_RUN_DIR / f"{model_name}_validation_threshold_sweep.csv", index=False
        )
        FINAL_RESULTS.append({
            "model": model_name,
            "split": "validation",
            **evaluate_predictions(VOC_SPLITS["validation"], validation_predictions, threshold),
        })
        FINAL_RESULTS.append({
            "model": model_name,
            "split": "test",
            **evaluate_predictions(VOC_SPLITS["test"], test_predictions, threshold),
        })
        test_predictions.to_csv(
            KERAS_RUN_DIR / f"{model_name}_esp32_test_predictions.csv", index=False
        )

    results_table = pd.DataFrame(FINAL_RESULTS)
    results_table.to_csv(KERAS_RUN_DIR / "float_int8_model_comparison.csv", index=False)
    display(results_table)

    drift = compare_prediction_drift(
        all_predictions["keras_mobilenetv2_float"][1],
        all_predictions["keras_mobilenetv2_uint8"][1],
    )
    drift.to_csv(KERAS_RUN_DIR / "keras_float_vs_uint8_drift.csv", index=False)
    display(drift.groupby("status").size().rename("detections").to_frame())
    if WRITE_C_ARRAY:
        write_decoder_metadata(
            SELECTED_THRESHOLDS["keras_mobilenetv2_uint8"]
        )
        print("Updated firmware metadata with validation-selected uint8 threshold")
else:
    print("Evaluation disabled.")


/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/home/joe/venvs/ESPclassifier/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,split,confidence_threshold,iou_threshold,true_positives,false_positives,false_negatives,precision,recall,f2,small_bird_recall,negative_images,false_positives_on_negative_images,false_positives_per_negative_image,mean_matched_iou
0,keras_mobilenetv2_float,validation,0.37,0.5,64,245,514,0.207120,0.110727,0.122091,0.045267,170,9,0.052941,0.646775
1,keras_mobilenetv2_float,test,0.37,0.5,99,625,141,0.136740,0.412500,0.293943,0.127273,104,229,2.201923,0.664074
2,keras_mobilenetv2_uint8,validation,0.37,0.5,52,234,526,0.181818,0.089965,0.100077,0.037037,170,13,0.076471,0.647050
3,keras_mobilenetv2_uint8,test,0.37,0.5,102,644,138,0.136729,0.425000,0.298945,0.100000,104,243,2.336538,0.683513


,detections
status,
lost_after_quantization,1215
matched,6585
new_after_quantization,1215


Updated firmware metadata with validation-selected uint8 threshold


## 11. Invocation-only benchmark and run contract


In [13]:
def benchmark_tflite_invocation(
    model_path: Path, warmup: int = 10, repeats: int = 100
) -> dict[str, Any]:
    interpreter = tf.lite.Interpreter(model_path=str(model_path), num_threads=max(1, os.cpu_count() or 1))
    interpreter.allocate_tensors()
    input_detail = interpreter.get_input_details()[0]
    shape = tuple(int(value) for value in input_detail["shape"])
    zeros = np.zeros(shape, dtype=np.float32)
    interpreter.set_tensor(input_detail["index"], quantize_array(zeros, input_detail))
    for _ in range(warmup):
        interpreter.invoke()
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        interpreter.invoke()
        timings.append((time.perf_counter() - start) * 1000)
    return {
        "model": str(model_path),
        "threads": max(1, os.cpu_count() or 1),
        "input_shape": shape,
        "size_mb": model_path.stat().st_size / 1_000_000,
        "mean_invoke_ms": float(np.mean(timings)),
        "p95_invoke_ms": float(np.percentile(timings, 95)),
    }


run_contract = {
    "schema_version": 1,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_run": str(SOURCE_RUN_DIR),
    "implementation": "plain TensorFlow/Keras; no TFLite Model Maker",
    "task": "single-class full-scene bird object detection",
    "class_map": {"0": "bird"},
    "architecture": {
        "backbone": "MobileNetV2 ImageNet transfer learning",
        "backbone_alpha": BACKBONE_ALPHA,
        "feature_layer": FEATURE_LAYER,
        "input_size": INPUT_SIZE,
        "grid_size": GRID_SIZE,
        "anchors": ANCHORS.tolist(),
        "raw_output_shape": [1, GRID_SIZE, GRID_SIZE, NUM_ANCHORS, 5],
    },
    "data": artifact_audit.to_dict(orient="records"),
    "quantization": {
        "precision": "full integer weights and activations",
        "input_dtype": "uint8",
        "output_dtype": "uint8",
        "representative_split": "ESP32 training calibration frames",
        "representative_images": len(VOC_SPLITS["calibration"][:REPRESENTATIVE_IMAGES]),
    },
    "selected_validation_thresholds": SELECTED_THRESHOLDS,
    "artifacts": {
        "weights": str(BEST_WEIGHTS),
        "float_tflite": str(FLOAT_TFLITE_PATH),
        "uint8_tflite": str(INT8_TFLITE_PATH),
        "model_header": str(FIRMWARE_DIR / "bird_detector_model.h"),
        "model_source": str(FIRMWARE_DIR / "bird_detector_model.cc"),
        "decoder_metadata": str(DECODER_METADATA_PATH),
    },
    "limitations": [
        "ESP32 firmware must implement output dequantization, decode, and NMS from the metadata contract.",
        "A successful desktop conversion does not prove the tensor arena fits the target ESP32 board.",
        "Register every operator listed in tflite_tensor_details.json in the TFLite Micro resolver.",
    ],
}
(KERAS_RUN_DIR / "run_contract.json").write_text(
    json.dumps(json_safe(run_contract), indent=2, allow_nan=False), encoding="utf-8"
)
print("Run contract:", KERAS_RUN_DIR / "run_contract.json")
print("Run benchmark_tflite_invocation(INT8_TFLITE_PATH) on the Raspberry Pi for a host benchmark.")


Run contract: /mnt/c/Users/Joe/Desktop/ESP_Classifier/esp32_Classifier/Models/Multi_Object_Detector/runs/bird_detector_20261026T012939Z/keras_micro_detector/run_contract.json
Run benchmark_tflite_invocation(INT8_TFLITE_PATH) on the Raspberry Pi for a host benchmark.


## 12. Recommended execution order

1. Select the Python 3.12 TensorFlow/WSL kernel and run sections 1-3. Confirm the source
   run and split audit. It should choose the latest run containing the complete VOC
   artifacts, not a newer empty directory.
2. Run sections 4-6. Existing `best.weights.h5` is restored automatically.
3. Run section 7 to create float and strict built-in-op `uint8` TFLite models.
4. Run section 8 to create `bird_detector_model.h`, `bird_detector_model.cc`, and the
   required decode metadata.
5. Run sections 9-10 to select confidence on ESP32 validation and evaluate ESP32 test
   exactly once.
6. Inspect `tflite_tensor_details.json`, register its operations in the ESP32 TFLite
   Micro resolver, and measure the required tensor arena on the actual board.

The existing YOLO `.tflite` artifacts remain untouched under the source run's
`exports/yolo26n` directory for comparison.
